# Hybrid CNN-SVM Bearing Fault Diagnosis

GitHub-friendly notebook for checking the environment, locating the CWRU dataset, and inspecting saved results from the CNN-SVM bearing fault diagnosis experiments.

Dataset: https://engineering.case.edu/bearingdatacenter/download-data-file  
Master thesis: https://www.repositorio.unicamp.br/acervo/detalhe/1434821  
Paper: Loading


## 1. Package check

Run this cell first. If packages are missing, install them in your Python environment before running the full experiments.

In [ ]:
import importlib.util

required_packages = [
    "numpy",
    "pandas",
    "scipy",
    "sklearn",
    "matplotlib",
    "tensorflow",
]

missing = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing:
    print("Missing packages:", missing)
    print("Install with:")
    print("pip install numpy pandas scipy scikit-learn matplotlib tensorflow jupyter")
else:
    print("All required packages were found.")


## 2. Paths

The dataset is not stored in this repository. Download it from CWRU and adjust the path below if necessary. The ablation output folder is optional; use it when you want to inspect saved results without retraining.

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
DATASET_DIR = ROOT / "dataset"
DATASET_ZIP = ROOT / "dataset-20210905T162441Z-001.zip"
ABLATION_DIR = ROOT / "ablation_outputs" / "ablation_outputs"

print("Current folder:", ROOT)
print("Dataset folder exists:", DATASET_DIR.exists(), DATASET_DIR)
print("Dataset zip exists:", DATASET_ZIP.exists(), DATASET_ZIP)
print("Ablation outputs exist:", ABLATION_DIR.exists(), ABLATION_DIR)


## 3. Inspect saved ablation results

This section reads the CSV files generated by the full experiment. It does not retrain the CNN models.

In [ ]:
import pandas as pd

summary_path = ABLATION_DIR / "ablation_summary.csv"

if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary.sort_values("mean_test_accuracy_percent", ascending=False).head(10))
else:
    print("No ablation_summary.csv found. Run the full experiment first or copy the saved outputs to:")
    print(ABLATION_DIR)


## 4. Plot selected configurations

The plot below summarizes the accuracy distribution for the CNN softmax baseline and selected CNN-SVM configurations, if the saved output file is available.

In [ ]:
import matplotlib.pyplot as plt

long_path = ABLATION_DIR / "ablation_accuracy_long.csv"

selected = {
    "cnn_softmax": "CNN softmax",
    "cnn_layer__pool2__linear__raw": "pool2 + linear SVM, raw",
    "cnn_layer__pool2__linear__standardized": "pool2 + linear SVM, std.",
    "cnn_layer__pool1__linear__standardized": "pool1 + linear SVM, std.",
    "cnn_layer__pool3__linear__raw": "pool3 + linear SVM, raw",
}

if long_path.exists():
    long_df = pd.read_csv(long_path)
    data = [long_df.loc[long_df["model_id"] == key, "test_accuracy_percent"].dropna().to_numpy() for key in selected]
    labels = list(selected.values())

    plt.figure(figsize=(9, 4))
    plt.boxplot(data, labels=labels, showmeans=False)
    plt.xticks(rotation=25, ha="right")
    plt.ylabel("Test accuracy (%)")
    plt.ylim(89, 100)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No ablation_accuracy_long.csv found. Copy saved outputs or run the full experiment.")


## 5. Full experiment

For the computationally expensive layer-wise ablation, use the full script/notebook from the paper material, for example:

```bash
python eai_ablation_experiments.py
```

Recommended: run the full ablation on a workstation or cluster and keep the generated CSV files, confusion matrices, predictions, histories, and figures. This avoids retraining every time the article figures need to be updated.